In [2]:
!pip install --quiet datasets
!huggingface-cli login --token blablabla

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `intent classifier` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `intent classifier`


In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 5.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/

In [3]:
from datasets import load_dataset
data = load_dataset('MBZUAI-Paris/Darija-SFT-Mixture')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/221M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/458285 [00:00<?, ? examples/s]

In [4]:
data

DatasetDict({
    train: Dataset({
        features: ['dataset', 'id', 'messages', 'direction', 'metadata'],
        num_rows: 458285
    })
})

In [4]:
data = data['train']

In [5]:
data

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata'],
    num_rows: 458285
})

In [6]:
set(data['dataset'])

{'9esa',
 'ElecMorocco2016',
 'ElecMorocco2016_few_shot',
 'ElecMorocco2016_multi_turns',
 'MAC',
 'MAC_few_shot',
 'MAC_multi_turns',
 'MArSum',
 'MArSum_few_shot',
 'MArSum_multi_turns',
 'MSAC',
 'MSAC_few_shot',
 'MSAC_multi_turns',
 'MSDA',
 'MSDA_few_shot',
 'MSDA_multi_turns',
 'MSMMG',
 'MSMMG_few_shot',
 'MSMMG_multi_turns',
 'MWQA',
 'MWQA_few_shot',
 'MWQA_multi_turns',
 'MYC',
 'MYC_few_shot',
 'MYC_multi_turns',
 'cot',
 'doda',
 'doda_few_shot',
 'doda_multi_turns',
 'flan_v2',
 'flores+',
 'flores+_few_shot',
 'flores+_multi_turns',
 'gpt4_alpaca',
 'hard_coded',
 'lima',
 'madar',
 'madar_few_shot',
 'madar_multi_turns',
 'nllb-seed',
 'nllb-seed_few_shot',
 'nllb-seed_multi_turns',
 'oasst1',
 'sharegpt',
 'wizardlm'}

In [24]:
from collections import Counter, defaultdict
import numpy as np
from datasets import Dataset

def _alloc_counts(total_n, counts_by_label):
    """
    Allocate total_n across labels proportionally to counts_by_label
    using largest remainder (Hamilton) method, ensuring sum == total_n.
    """
    labels = list(counts_by_label.keys())
    total = sum(counts_by_label.values())
    if total == 0:
        return {k: 0 for k in labels}

    # ideal fractional allocations
    ideal = {k: total_n * (counts_by_label[k] / total) for k in labels}
    base = {k: int(np.floor(ideal[k])) for k in labels}
    rem = total_n - sum(base.values())

    # distribute remaining by largest fractional parts
    frac = sorted(labels, key=lambda k: (ideal[k] - base[k]), reverse=True)
    for k in frac[:rem]:
        base[k] += 1

    return base

def stratified_subsets_disjoint(
    ds: Dataset,
    sizes=(50_000, 70_000, 100_000),
    label_col="dataset",
    seed=42,
):
    """
    Build multiple DISJOINT stratified subsets from ds based on label_col.
    Returns: list_of_subsets, remaining_ds
    """
    n_total = len(ds)
    if sum(sizes) > n_total:
        raise ValueError(f"Sum(sizes)={sum(sizes)} > dataset size={n_total}")

    labels = ds[label_col]
    # map label -> list of indices
    idx_by_label = defaultdict(list)
    for i, lab in enumerate(labels):
        idx_by_label[lab].append(i)

    # shuffle indices within each label for randomness
    rng = np.random.default_rng(seed)
    for lab in idx_by_label:
        rng.shuffle(idx_by_label[lab])

    subsets = []
    remaining_idx_by_label = {lab: idxs[:] for lab, idxs in idx_by_label.items()}

    for step, n in enumerate(sizes, start=1):
        # current availability counts
        avail_counts = {lab: len(idxs) for lab, idxs in remaining_idx_by_label.items()}

        # proportional allocation
        want = _alloc_counts(n, avail_counts)

        # if any label doesn't have enough (can happen after earlier draws),
        # take what exists and redistribute the shortfall to others with slack
        take = {lab: min(want[lab], avail_counts[lab]) for lab in want}
        short = n - sum(take.values())

        if short > 0:
            # labels with remaining slack
            slack = {lab: avail_counts[lab] - take[lab] for lab in take if avail_counts[lab] - take[lab] > 0}
            # redistribute shortfall proportionally to slack
            if sum(slack.values()) == 0:
                raise RuntimeError("Not enough remaining samples to satisfy requested size (unexpected).")

            add = _alloc_counts(short, slack)
            for lab, extra in add.items():
                take[lab] += min(extra, avail_counts[lab] - take[lab])

            # still short? fill greedily from any remaining
            short2 = n - sum(take.values())
            if short2 > 0:
                for lab in sorted(take.keys(), key=lambda k: (avail_counts[k] - take[k]), reverse=True):
                    if short2 == 0:
                        break
                    can = avail_counts[lab] - take[lab]
                    if can > 0:
                        d = min(can, short2)
                        take[lab] += d
                        short2 -= d
                if short2 > 0:
                    raise RuntimeError("Could not allocate enough samples (unexpected).")

        # materialize indices for this subset
        subset_indices = []
        for lab, k in take.items():
            if k > 0:
                subset_indices.extend(remaining_idx_by_label[lab][:k])
                remaining_idx_by_label[lab] = remaining_idx_by_label[lab][k:]

        # shuffle subset indices (so it's not grouped by label)
        rng.shuffle(subset_indices)

        subsets.append(ds.select(subset_indices))

    # build remaining dataset
    remaining_indices = []
    for lab, idxs in remaining_idx_by_label.items():
        remaining_indices.extend(idxs)
    rng.shuffle(remaining_indices)
    remaining_ds = ds.select(remaining_indices)

    return subsets, remaining_ds


# ---- usage ----
(subdata_1, subdata_2, subdata_3), rest = stratified_subsets_disjoint(
    data,
    sizes=(55_000, 80_000, 113_000),
    label_col="dataset",
    seed=123,
)

print(len(subdata_1), len(subdata_2), len(subdata_3), len(rest))


55000 80000 113000 210285


In [7]:
from random import choice
a = choice(subdata_1)['messages']
for x in a:
  print(x['role'])
  print(x['content'])
  print('--------------')

user
(80/100) * 60 - (4/5) * 25
48 - 20 = 28
شحال 80% من 60 كبر من 4/5 من 25؟
الخيارات:
(A) 15
(B) 28
(C) 16
(D) 12
(E) 73
(B)

A = 7500(26/25)^4 = 8774
إلا تسلفتي 7500 درهم بالفايدة المركبة بنسبة 4% فالعام، شحال غادي تخلص من بعد 4 سنين؟
الخيارات:
(A) 3377
(B) 2688
(C) 2688
(D) 8774
(E) 1268
(D)

1/14 - 1/28 = 1/28
28 دقيقة

--------------
assistant
واحد الروبيني كيقدر يعمر خزان فـ14 دقيقة، بينما الخزان لما كيتعمر كيقدر يتفرغ من تقب فـ28 دقيقة. إلا حليتي بجوج الروبينيات، فوقتاش غادي يتعمر الخزان؟
الخيارات:
(A) 17
(B) 16
(C) 28
(D) 13
(E) 12
(C)
--------------


In [25]:
from datasets import load_dataset
from transformers import AutoTokenizer
from unsloth.chat_templates import get_chat_template


# 2. Load the base tokenizer for Gemma-3
tok = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")

# 3. Wrap it with the gemma-3 chat template
tok = get_chat_template(
    tok,
    chat_template="gemma-3",            # select the gemma-3 template
)

# 4. Define a formatting fn that applies the template to each conversation
def format_for_gemma3(examples):
    convos = examples["messages"]  # list[dict(role,content)] per example
    try:
      texts = [
          tok.apply_chat_template(
              convo,
              tokenize=False,             # return raw string
              add_generation_prompt=False # no extra “assistant:” at end
          )
          for convo in convos
      ]
      return {"text": texts}
    except:
      return {"text": [None]*len(convos)}

# 5. Map it over your dataset
#ds = ds.map(format_for_gemma3, batched=True)

# Now ds["text"] holds strings like:
# <s>user: …<endofturn>model: …<endofturn>user: …<endofturn>model:


In [26]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")

In [15]:
subdata_1[5]['messages']

[{'content': 'شنو هو الإحساس ديال هاد الجملة؟\nالعبارة: الله يبارك فيك\n الإحتمالات:\n-سلبي\n-ايجابي\n-ماكينش إحساس',
  'role': 'user'},
 {'content': 'ايجابي', 'role': 'assistant'}]

In [16]:
subdata_1[28000]['messages']

[{'content': 'كتب هادشي بالحروف ديال الفرنسية:\nلمغاربا كاياكلو كسكسو كول جمعا',
  'role': 'user'},
 {'content': 'lmgharba kayaklou ksksou koul jm3a', 'role': 'assistant'}]

In [27]:
subdata_1 = subdata_1.map(format_for_gemma3, batched=True)
subdata_2 = subdata_2.map(format_for_gemma3, batched=True)
subdata_3 = subdata_3.map(format_for_gemma3, batched=True)

Map:   0%|          | 0/55000 [00:00<?, ? examples/s]

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/113000 [00:00<?, ? examples/s]

In [28]:
subdata_1 = subdata_1.filter(lambda example: example["text"] is not None)
subdata_2 = subdata_2.filter(lambda example: example["text"] is not None)
subdata_3 = subdata_3.filter(lambda example: example["text"] is not None)
len(subdata_1), len(subdata_2), len(subdata_3)

Filter:   0%|          | 0/55000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/80000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/113000 [00:00<?, ? examples/s]

(53000, 74000, 104000)

In [29]:
ds1 = subdata_1.map(lambda x: {"tokens": tokenizer(x["text"])})

Map:   0%|          | 0/53000 [00:00<?, ? examples/s]

In [30]:
ds1_ok = ds1.filter(lambda example: len(example["tokens"]["input_ids"]) <= 2048)
ds1_ok

Filter:   0%|          | 0/53000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 52843
})

In [31]:
ds1_too_long = ds1.filter(lambda example: len(example["tokens"]["input_ids"]) > 2048)
ds1_too_long

Filter:   0%|          | 0/53000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 157
})

In [32]:
subdata_1_ok = ds1_ok.select(range(50000))
subdata_1_ok

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 50000
})

In [37]:
subdata_1_ok.push_to_hub('GemMaroc/AtlasChat-data-50k-randomlysampled-darija')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   2%|1         | 1.08MB / 68.0MB            

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   6%|5         | 3.71MB / 67.4MB            

CommitInfo(commit_url='https://huggingface.co/datasets/GemMaroc/AtlasChat-data-50k-randomlysampled-darija/commit/f2f66542e5f7001a27bb4f131a21621a60bd0ff8', commit_message='Upload dataset', commit_description='', oid='f2f66542e5f7001a27bb4f131a21621a60bd0ff8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/GemMaroc/AtlasChat-data-50k-randomlysampled-darija', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GemMaroc/AtlasChat-data-50k-randomlysampled-darija'), pr_revision=None, pr_num=None)

In [36]:
choice(subdata_1_ok)['text']

'<bos><start_of_turn>user\nشنو هي التقنيات الموسيقية لي مستعملين ف "يانيس وفلافيو: قصة حب موسيقية" باش يبرزو الفروقات الثقافية بين الشخصيتين؟\r\n\r\n\\documentclass{article}\r\n\r\n% Use packages\r\n\\usepackage[utf8]{inputenc}\r\n\\usepackage{graphicx}\r\n\\usepackage{geometry}\r\n\\usepackage{multicol}\r\n\\usepackage{amsmath}\r\n\r\n% Set page margins\r\n\\geometry{a4paper, total={170mm,257mm}, left=20mm, top=20mm}\r\n\r\n% Set font\r\n\\renewcommand{\\familydefault}{\\sfdefault}\r\n\r\n% Set title and author\r\n\\title{يانيس وفلافيو: قصة حب موسيقية}\r\n\\author{المؤلف: [سميتك]}\r\n\r\n\\begin{document}\r\n\r\n\\maketitle\r\n\r\n% Set sections\r\n\\section{المقدمة}\r\n\r\nيانيس كان مهندس معماري يوناني ماهر سافر لمعرض فني دولي فين تلاقى مع فلافيو، فنان إيطالي مشهور. مع الوقت، يانيس تعجب بطريقة فلافيو الفريدة فالفن، شخصيتو الجذابة، وعبقريتو الفنية. ولكن، الحبايب واجهو فروقات ثقافية فيما يخص نظرتهم للعائلة، الطبخ، والموسيقى. هاد التحفة الموسيقية كتهدف لتصوير القصة المعقدة ديال قصة حب ي

In [38]:
ds2 = subdata_2.map(lambda x: {"tokens": tokenizer(x["text"])})
ds2_ok = ds2.filter(lambda example: len(example["tokens"]["input_ids"]) <= 2048)
ds2_ok

Map:   0%|          | 0/74000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/74000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 73741
})

In [39]:
subdata_2_ok = ds2_ok.select(range(70000))
subdata_2_ok.push_to_hub('GemMaroc/AtlasChat-data-70k-randomlysampled-darija')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          |  529kB / 95.3MB            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   4%|3         | 3.71MB / 93.7MB            

CommitInfo(commit_url='https://huggingface.co/datasets/GemMaroc/AtlasChat-data-70k-randomlysampled-darija/commit/b77446f2dd44b9e5af847f1573037b3c7f614ca9', commit_message='Upload dataset', commit_description='', oid='b77446f2dd44b9e5af847f1573037b3c7f614ca9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/GemMaroc/AtlasChat-data-70k-randomlysampled-darija', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GemMaroc/AtlasChat-data-70k-randomlysampled-darija'), pr_revision=None, pr_num=None)

In [40]:
ds3 = subdata_3.map(lambda x: {"tokens": tokenizer(x["text"])})
ds3_ok = ds3.filter(lambda example: len(example["tokens"]["input_ids"]) <= 2048)
ds3_ok

Map:   0%|          | 0/104000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/104000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 103625
})

In [41]:
subdata_3_ok = ds3_ok.select(range(100000))
subdata_3_ok.push_to_hub('GemMaroc/AtlasChat-data-100k-randomlysampled-darija')

Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          |  529kB / 90.9MB            

...

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   8%|8         | 7.45MB / 89.6MB            

CommitInfo(commit_url='https://huggingface.co/datasets/GemMaroc/AtlasChat-data-100k-randomlysampled-darija/commit/10853b8f69fa11e8aa32be948041af89dcefb9a6', commit_message='Upload dataset', commit_description='', oid='10853b8f69fa11e8aa32be948041af89dcefb9a6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/GemMaroc/AtlasChat-data-100k-randomlysampled-darija', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GemMaroc/AtlasChat-data-100k-randomlysampled-darija'), pr_revision=None, pr_num=None)

# now qwen

In [52]:
(subdata_1, subdata_2, subdata_3), rest = stratified_subsets_disjoint(
    data,
    sizes=(60_000, 90_000, 120_000),
    label_col="dataset",
    seed=123,
)

print(len(subdata_1), len(subdata_2), len(subdata_3), len(rest))

60000 90000 120000 188285


In [53]:
from datasets import load_dataset
from transformers import AutoTokenizer
from unsloth.chat_templates import get_chat_template


# 2. Load the base tokenizer for Gemma-3
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct")

# 3. Wrap it with the gemma-3 chat template
tok = get_chat_template(
    tok,
    chat_template="qwen2.5",            # select the gemma-3 template
)

# 4. Define a formatting fn that applies the template to each conversation
def format_for_gemma3(examples):
    convos = examples["messages"]  # list[dict(role,content)] per example
    try:
      texts = [
          tok.apply_chat_template(
              convo,
              tokenize=False,             # return raw string
              add_generation_prompt=False # no extra “assistant:” at end
          )
          for convo in convos
      ]
      return {"text": texts}
    except:
      return {"text": [None]*len(convos)}


In [54]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct")

In [55]:
subdata_1 = subdata_1.map(format_for_gemma3, batched=True)
subdata_2 = subdata_2.map(format_for_gemma3, batched=True)
subdata_3 = subdata_3.map(format_for_gemma3, batched=True)

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

In [56]:
subdata_1 = subdata_1.filter(lambda example: example["text"] is not None)
subdata_2 = subdata_2.filter(lambda example: example["text"] is not None)
subdata_3 = subdata_3.filter(lambda example: example["text"] is not None)
len(subdata_1), len(subdata_2), len(subdata_3)

Filter:   0%|          | 0/60000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/90000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

(60000, 90000, 120000)

In [57]:
ds1 = subdata_1.map(lambda x: {"tokens": tokenizer(x["text"])})
ds1_ok = ds1.filter(lambda example: len(example["tokens"]["input_ids"]) <= 2048)
ds1_ok

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/60000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 54802
})

In [58]:
subdata_1_ok = ds1_ok.select(range(50000))
subdata_1_ok.push_to_hub('GemMaroc/AtlasChat-data-50k-randomlysampled-darija-qwen-template')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|1         |  533kB / 47.9MB            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   8%|7         | 3.74MB / 48.0MB            

CommitInfo(commit_url='https://huggingface.co/datasets/GemMaroc/AtlasChat-data-50k-randomlysampled-darija-qwen-template/commit/99758573530a4bae1dfa459bea0bf4d57a5a6245', commit_message='Upload dataset', commit_description='', oid='99758573530a4bae1dfa459bea0bf4d57a5a6245', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/GemMaroc/AtlasChat-data-50k-randomlysampled-darija-qwen-template', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GemMaroc/AtlasChat-data-50k-randomlysampled-darija-qwen-template'), pr_revision=None, pr_num=None)

In [59]:
ds2 = subdata_2.map(lambda x: {"tokens": tokenizer(x["text"])})
ds2_ok = ds2.filter(lambda example: len(example["tokens"]["input_ids"]) <= 2048)
ds2_ok

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/90000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 82229
})

In [60]:
subdata_2_ok = ds2_ok.select(range(70000))
subdata_2_ok.push_to_hub('GemMaroc/AtlasChat-data-70k-randomlysampled-darija-qwen-template')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          |  533kB / 67.2MB            

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          |  533kB / 66.7MB            

CommitInfo(commit_url='https://huggingface.co/datasets/GemMaroc/AtlasChat-data-70k-randomlysampled-darija-qwen-template/commit/dca4a5b84fcad9a50c53e02de59e982115870cc9', commit_message='Upload dataset', commit_description='', oid='dca4a5b84fcad9a50c53e02de59e982115870cc9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/GemMaroc/AtlasChat-data-70k-randomlysampled-darija-qwen-template', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GemMaroc/AtlasChat-data-70k-randomlysampled-darija-qwen-template'), pr_revision=None, pr_num=None)

In [61]:
ds3 = subdata_3.map(lambda x: {"tokens": tokenizer(x["text"])})
ds3_ok = ds3.filter(lambda example: len(example["tokens"]["input_ids"]) <= 2048)
ds3_ok

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Dataset({
    features: ['dataset', 'id', 'messages', 'direction', 'metadata', 'text', 'tokens'],
    num_rows: 109614
})

In [62]:
subdata_3_ok = ds3_ok.select(range(100000))
subdata_3_ok.push_to_hub('GemMaroc/AtlasChat-data-100k-randomlysampled-darija-qwen-template')

Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          |  534kB / 64.6MB            

...

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   6%|5         | 3.73MB / 64.1MB            

CommitInfo(commit_url='https://huggingface.co/datasets/GemMaroc/AtlasChat-data-100k-randomlysampled-darija-qwen-template/commit/139f57c5861245950472e6e9aef3d37f65caaf5d', commit_message='Upload dataset', commit_description='', oid='139f57c5861245950472e6e9aef3d37f65caaf5d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/GemMaroc/AtlasChat-data-100k-randomlysampled-darija-qwen-template', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GemMaroc/AtlasChat-data-100k-randomlysampled-darija-qwen-template'), pr_revision=None, pr_num=None)